In [1]:
import random
import typing
from dataclasses import dataclass

import simpy
import simpy.resources.resource

In [2]:
def singleton(cls: typing.Any):
    instances = {}

    def get_instance(*args: typing.Any, **kwargs: typing.Any) -> typing.Any:
        if cls not in instances:
            instances[cls] = cls(*args, **kwargs)
        return instances[cls]

    return get_instance

In [3]:
class FloatRange:
    def __init__(self, start: float, end: float) -> None:
        self._start = start
        self._end = end

    def get_random(self) -> float:
        return random.uniform(self._start, self._end)

In [4]:
class IHealthStatus(typing.Protocol):
    @property
    def name(self) -> str: ...

    @property
    def is_contagious(self) -> bool: ...

    @property
    def can_work(self) -> bool: ...

    @property
    def emoji(self) -> str: ...


@singleton
class Healthy(IHealthStatus):
    @property
    def name(self) -> str:
        return "healthy"

    @property
    def is_contagious(self) -> bool:
        return False

    @property
    def can_work(self) -> bool:
        return True

    @property
    def emoji(self) -> str:
        return "💚"


@singleton
class Incubating(IHealthStatus):
    @property
    def name(self) -> str:
        return "incubating"

    @property
    def is_contagious(self) -> bool:
        return True

    @property
    def can_work(self) -> bool:
        return True

    @property
    def emoji(self) -> str:
        return "🟡"


@singleton
class Sick(IHealthStatus):
    @property
    def name(self) -> str:
        return "sick"

    @property
    def is_contagious(self) -> bool:
        return True

    @property
    def can_work(self) -> bool:
        return False

    @property
    def emoji(self) -> str:
        return "🔴"


@singleton
class Recovered(IHealthStatus):
    @property
    def name(self) -> str:
        return "recovered"

    @property
    def is_contagious(self) -> bool:
        return False

    @property
    def can_work(self) -> bool:
        return True

    @property
    def emoji(self) -> str:
        return "💙"

In [5]:
class IHealthModifier(typing.Protocol):
    @property
    def name(self) -> str: ...

    def modify_infection_probability(self, base_probability: float) -> float: ...

    def modify_sick_duration(self, base_duration: float) -> float: ...


@singleton
class Mask(IHealthModifier):
    @property
    def name(self) -> str:
        return "😷"

    def modify_infection_probability(self, base_probability: float) -> float:
        return base_probability * 0.5

    def modify_sick_duration(self, base_duration: float) -> float:
        return base_duration


@singleton
class Vaccine(IHealthModifier):
    @property
    def name(self) -> str:
        return "💉"

    def modify_infection_probability(self, base_probability: float) -> float:
        return base_probability * 0.2

    def modify_sick_duration(self, base_duration: float) -> float:
        return base_duration * 0.6

In [6]:
class IInfection(typing.Protocol):
    def get_infection_probability(self, duration: float, sick_count: int) -> float: ...

    def get_incubation_duration(self) -> float: ...

    def get_sick_duration(self) -> float: ...


class Covid(IInfection):
    def __init__(
        self,
        infection_rate: float,
        incubation_range: FloatRange,
        sick_range: FloatRange,
    ) -> None:
        self._infection_rate = infection_rate
        self._incubation_range = incubation_range
        self._sick_range = sick_range

    def get_infection_probability(self, duration: float, sick_count: int) -> float:
        if sick_count == 0:
            return 0.0
        base = self._infection_rate * (duration / 10.0) * sick_count
        return min(base, 1.0)

    def get_incubation_duration(self) -> float:
        return self._incubation_range.get_random()

    def get_sick_duration(self) -> float:
        return self._sick_range.get_random()

In [7]:
class Room:
    def __init__(
        self,
        env: simpy.Environment,
        name: str,
        capacity: int,
        visit_duration: FloatRange,
        ventilation_quality: float,
    ) -> None:
        self._env = env
        self._name = name
        self._capacity = capacity
        self._visit_duration = visit_duration
        self._ventilation_quality = ventilation_quality
        self._resource = simpy.Resource(env, capacity)

    @property
    def name(self) -> str:
        return self._name

    @property
    def ventilation_quality(self) -> float:
        return self._ventilation_quality

    def get_request(self) -> simpy.resources.resource.Request:
        return self._resource.request()

    def get_visit_duration(self) -> float:
        return self._visit_duration.get_random()

In [8]:
class HealthState:
    def __init__(
        self,
        infection: IInfection,
        modifiers: typing.Sequence[IHealthModifier] = (),
        is_initially_infected: bool = False
    ) -> None:
        self._infection = infection
        self._modifiers = list(modifiers)
        self._status = Healthy()
        self._transition_time: float | None = None
        self._next_status: IHealthStatus | None = None

        if is_initially_infected:
            self._status = Incubating()
            self._schedule_transition(0.0, Sick())

    @property
    def status(self) -> IHealthStatus:
        return self._status

    @property
    def modifiers(self) -> list[IHealthModifier]:
        return self._modifiers

    def update(self, current_time: float) -> None:
        if self._transition_time and current_time >= self._transition_time:
            self._do_transition(current_time)

    def infect(self, current_time: float) -> None:
        if self._status is Healthy():
            self._status = Incubating()
            self._schedule_transition(current_time, Sick())

    def get_infection_probability(
        self,
        duration: float,
        sick_count: int,
        ventilation: float,
    ) -> float:
        prob = self._infection.get_infection_probability(duration, sick_count)
        prob *= ventilation
        for modifier in self._modifiers:
            prob = modifier.modify_infection_probability(prob)
        return prob

    def _do_transition(self, current_time: float) -> None:
        if self._next_status is None:
            return
        self._status = self._next_status
        if self._status is Sick():
            self._schedule_transition(current_time, Recovered())
        else:
            self._transition_time = None
            self._next_status = None

    def _schedule_transition(self, current_time: float, next_status: IHealthStatus) -> None:
        if next_status is Sick():
            duration = self._infection.get_incubation_duration()
        elif next_status is Recovered():
            duration = self._infection.get_sick_duration()
            for modifier in self._modifiers:
                duration = modifier.modify_sick_duration(duration)
        else:
            return
        self._transition_time = current_time + duration
        self._next_status = next_status


In [9]:
class Employee:
    def __init__(
        self,
        env: simpy.Environment,
        name: str,
        health: HealthState,
        room_visit_weights: typing.Mapping[str, float],
        in_quarantine: bool = False,
    ) -> None:
        self._env = env
        self._name = name
        self._health = health
        self._room_visit_weights = dict(room_visit_weights)
        self.in_quarantine = in_quarantine

    @property
    def status(self) -> IHealthStatus:
        return self._health.status

    @property
    def name(self) -> str:
        return self._name

    @property
    def room_visit_weights(self) -> dict[str, float]:
        return self._room_visit_weights

    @property
    def health_modifiers(self) -> list[IHealthModifier]:
        return self._health.modifiers

    def update_health(self) -> None:
        self._health.update(self._env.now)

    def quarantine(self) -> None:
        self.in_quarantine = True

    def test_for_virus(self) -> bool:
        is_sick = self.status.is_contagious
        if random.random() < 0.9:
            return is_sick
        else:
            return not is_sick

    def visit_room(
        self,
        room: Room,
        all_employees: typing.Sequence["Employee"],
    ) -> typing.Iterator[simpy.Event]:
        with room.get_request() as req:
            yield req
            duration = room.get_visit_duration()
            yield self._env.timeout(duration)

            if self.status is Healthy() and not self.in_quarantine:
                self._try_get_infected(all_employees, duration, room.ventilation_quality)

    def _try_get_infected(
        self,
        employees: typing.Sequence["Employee"],
        duration: float,
        ventilation: float,
    ) -> None:
        sick_count = sum(
            1 for emp in employees
            if emp is not self
            and emp.status.is_contagious
            and not emp.in_quarantine
        )

        if sick_count == 0:
            return

        prob = self._health.get_infection_probability(duration, sick_count, ventilation)

        if random.random() < prob:
            self._health.infect(self._env.now)


In [10]:
@dataclass
class SimulationStats:
    time: list[float]
    healthy: list[int]
    incubating: list[int]
    sick: list[int]
    recovered: list[int]
    in_quarantine: list[int]

    def __init__(self):
        self.time = []
        self.healthy = []
        self.incubating = []
        self.sick = []
        self.recovered = []
        self.in_quarantine = []

    def record(self, env_time: float, employees: typing.Sequence[Employee]) -> None:
        self.time.append(env_time)
        self.healthy.append(sum(1 for e in employees if e.status is Healthy()))
        self.incubating.append(sum(1 for e in employees if e.status is Incubating()))
        self.sick.append(sum(1 for e in employees if e.status is Sick()))
        self.recovered.append(sum(1 for e in employees if e.status is Recovered()))
        self.in_quarantine.append(sum(1 for e in employees if e.in_quarantine))

In [11]:
class Office:
    def __init__(
        self,
        env: simpy.Environment,
        rooms: typing.Sequence[Room],
        employees: typing.Sequence[Employee],
        quarantine_threshold: float = 0.3,
        testing_frequency: float = 0,
    ) -> None:
        self._env = env
        self._rooms = list(rooms)
        self._employees = list(employees)
        self._quarantine_threshold = quarantine_threshold
        self._testing_frequency = testing_frequency
        self._quarantine_activated = False
        self._stats = SimulationStats()

    @property
    def employees(self) -> list[Employee]:
        return self._employees

    @property
    def stats(self) -> SimulationStats:
        return self._stats

    @property
    def testing_frequency(self) -> float:
        return self._testing_frequency

    def get_room(self, name: str) -> Room | None:
        for room in self._rooms:
            if room.name == name:
                return room
        return None

    def check_quarantine(self) -> None:
        sick_ratio = sum(1 for e in self._employees if e.status is Sick()) / len(self._employees)

        if sick_ratio >= self._quarantine_threshold and not self._quarantine_activated:
            self._quarantine_activated = True
            for emp in self._employees:
                emp.quarantine()

    def run_testing(self) -> typing.Iterator[simpy.Event]:
        if self._testing_frequency == 0:
            return

        while True:
            yield self._env.timeout(self._testing_frequency)

            for emp in self._employees:
                if emp.test_for_virus() and not emp.in_quarantine:
                    emp.quarantine()

    def collect_stats(self) -> typing.Iterator[simpy.Event]:
        while True:
            self._stats.record(self._env.now, self._employees)
            self.check_quarantine()
            yield self._env.timeout(5)

In [12]:
def work_day(env: simpy.Environment, employee: Employee, office: Office) -> typing.Iterator[simpy.Event]:
    for _ in range(10):
        yield env.timeout(random.uniform(20, 40))
        employee.update_health()

        if employee.in_quarantine or not employee.status.can_work:
            if employee.status is Sick() and not employee.in_quarantine:
                return

        rooms_list = []
        weights_list = []

        for room_name, weight in employee.room_visit_weights.items():
            room = office.get_room(room_name)
            if room:
                rooms_list.append(room)
                weights_list.append(weight)

        rooms_list.append(None)
        weights_list.append(0.5)

        chosen_room = random.choices(rooms_list, weights=weights_list)[0]

        if chosen_room:
            yield env.process(employee.visit_room(chosen_room, office.employees))


In [13]:
def create_office_simulation(
    env: simpy.Environment,
    scenario: typing.Literal[
        "baseline",
        "partial_protection",
        "high_protection",
        "poor_ventilation",
        "aggressive_testing",
        "mixed_conditions"
    ],
) -> Office:
    infection_params = {
        "baseline": {"rate": 0.05, "incubation": (24, 48), "sick": (120, 240)},
        "aggressive": {"rate": 0.08, "incubation": (12, 36), "sick": (96, 180)},
        "mild": {"rate": 0.03, "incubation": (36, 60), "sick": (144, 288)},
    }

    if scenario in ["baseline", "partial_protection", "high_protection"]:
        inf_params = infection_params["baseline"]
    elif scenario in ["aggressive_testing", "poor_ventilation"]:
        inf_params = infection_params["aggressive"]
    else:
        inf_params = infection_params["mild"]

    infection = Covid(
        infection_rate=inf_params["rate"],
        incubation_range=FloatRange(*inf_params["incubation"]),
        sick_range=FloatRange(*inf_params["sick"])
    )

    ventilation_quality = {
        "baseline": 1.0,
        "partial_protection": 1.0,
        "high_protection": 0.8,
        "poor_ventilation": 1.5,
        "aggressive_testing": 1.0,
        "mixed_conditions": 1.2,
    }.get(scenario, 1.0)

    rooms = [
        Room(env, "Conference Room", 10, FloatRange(15, 45), ventilation_quality),
        Room(env, "Kitchen", 5, FloatRange(5, 15), ventilation_quality * 1.2),
        Room(env, "Open Space", 30, FloatRange(30, 90), ventilation_quality * 0.8),
        Room(env, "Meeting Room", 6, FloatRange(20, 60), ventilation_quality),
    ]

    room_weights = {
        "Conference Room": 0.3,
        "Kitchen": 0.5,
        "Open Space": 0.8,
        "Meeting Room": 0.2,
    }

    employees = []
    num_employees = 50

    protection_configs = {
        "baseline": {"mask_ratio": 0.0, "vaccine_ratio": 0.0},
        "partial_protection": {"mask_ratio": 0.5, "vaccine_ratio": 0.0},
        "high_protection": {"mask_ratio": 0.5, "vaccine_ratio": 0.7},
        "poor_ventilation": {"mask_ratio": 0.2, "vaccine_ratio": 0.1},
        "aggressive_testing": {"mask_ratio": 0.3, "vaccine_ratio": 0.5},
        "mixed_conditions": {"mask_ratio": 0.6, "vaccine_ratio": 0.4},
    }

    config = protection_configs.get(scenario, protection_configs["baseline"])

    initial_infected = 3 if scenario != "aggressive_testing" else 5

    for i in range(num_employees):
        modifiers = []

        if random.random() < config["mask_ratio"]:
            modifiers.append(Mask())

        if random.random() < config["vaccine_ratio"]:
            modifiers.append(Vaccine())

        is_infected = i < initial_infected

        health = HealthState(
            infection=infection,
            modifiers=modifiers,
            is_initially_infected=is_infected
        )

        employee = Employee(
            env=env,
            name=f"Emp-{i + 1:02d}",
            health=health,
            room_visit_weights=room_weights
        )

        employees.append(employee)

    quarantine_threshold = {
        "baseline": 0.3,
        "partial_protection": 0.3,
        "high_protection": 0.4,
        "poor_ventilation": 0.25,
        "aggressive_testing": 0.5,
        "mixed_conditions": 0.35,
    }.get(scenario, 0.3)

    testing_frequency = {
        "baseline": 0,
        "partial_protection": 0,
        "high_protection": 0,
        "poor_ventilation": 0,
        "aggressive_testing": 48,
        "mixed_conditions": 72,
    }.get(scenario, 0)

    office = Office(
        env=env,
        rooms=rooms,
        employees=employees,
        quarantine_threshold=quarantine_threshold,
        testing_frequency=testing_frequency
    )

    for employee in employees:
        env.process(work_day(env, employee, office))

    env.process(office.collect_stats())

    if testing_frequency > 0:
        env.process(office.run_testing())

    return office

In [14]:
def print_dashboard(office: Office, scenario: str, duration: float) -> None:
    stats = office.stats
    employees = office.employees
    total = len(employees)

    counts = {
        'H': sum(1 for e in employees if e.status is Healthy()),
        'I': sum(1 for e in employees if e.status is Incubating()),
        'S': sum(1 for e in employees if e.status is Sick()),
        'R': sum(1 for e in employees if e.status is Recovered()),
    }

    infected = counts['I'] + counts['S'] + counts['R']
    peak_sick = max(stats.sick) if stats.sick else 0
    peak_time = stats.time[stats.sick.index(peak_sick)] if peak_sick > 0 else 0

    print()
    print("  " + scenario.upper().replace("_", " "))
    print()

    def bar(n: int) -> str:
        width = int(n / total * 25)
        return "#" * width

    print("  Statistics")
    print("  " + "=" * 50)
    print(f"  H {counts['H']:2d}  {bar(counts['H'])}")
    print(f"  I {counts['I']:2d}  {bar(counts['I'])}")
    print(f"  S {counts['S']:2d}  {bar(counts['S'])}")
    print(f"  R {counts['R']:2d}  {bar(counts['R'])}")
    print()

    print(f"  Infected: {infected}/{total} ({infected / total * 100:.0f}%)")
    print(f"  Peak:     {peak_sick} at t={peak_time:.0f}")
    print()

    if len(stats.healthy) > 10:
        print("  Timeline Heatmap")
        print("  " + "=" * 50)

        width = 48
        n = len(stats.healthy)
        step = max(1, n // width)

        h_data = [stats.healthy[i] for i in range(0, n, step)][:width]
        i_data = [stats.incubating[i] for i in range(0, n, step)][:width]
        s_data = [stats.sick[i] for i in range(0, n, step)][:width]
        r_data = [stats.recovered[i] for i in range(0, n, step)][:width]

        intensity = [' ', '·', ':', '░', '▒', '▓', '█']

        def get_intensity(val: int) -> str:
            idx = min(len(intensity) - 1, int(val / total * len(intensity)))
            return intensity[idx]

        print("  H |", end="")
        for val in h_data:
            print(get_intensity(val), end="")
        print("|")

        print("  I |", end="")
        for val in i_data:
            print(get_intensity(val), end="")
        print("|")

        print("  S |", end="")
        for val in s_data:
            print(get_intensity(val), end="")
        print("|")

        print("  R |", end="")
        for val in r_data:
            print(get_intensity(val), end="")
        print("|")

        print("    " + "-" * 48)

        t_quarter = duration / 4
        print(
            f"    0",
            f"{t_quarter:.0f}",
            f"{t_quarter * 2:.0f}",
            f"{t_quarter * 3:.0f}",
            f"{duration:.0f}",
            sep=" " * 9,
        )
        print()

        print("  Key Events")
        print("  " + "=" * 50)

        events = []

        for i, s in enumerate(stats.sick):
            if s > 0:
                events.append((stats.time[i], "First sick", "●"))
                break

        events.append((peak_time, f"Peak sick ({peak_sick})", "▲"))

        half_infected = total // 2
        for i in range(len(stats.sick)):
            inf = stats.sick[i] + stats.recovered[i] + stats.incubating[i]
            if inf >= half_infected:
                events.append((stats.time[i], "50% infected", "◆"))
                break

        for i, r in enumerate(stats.recovered):
            if r > 0:
                events.append((stats.time[i], "First recovery", "○"))
                break

        events.sort()

        for t, desc, symbol in events:
            pos = int((t / duration) * 46)
            spaces = " " * pos
            print(f"  {spaces}{symbol} {desc} (t={t:.0f})")

        print("  " + "-" * 50)
        print()
        print("  Intensity: ' ' = None   '·' = Low   '█' = High")

    print()

In [15]:
def run_simulation(
    scenario: typing.Literal[
        "baseline",
        "partial_protection",
        "high_protection",
        "poor_ventilation",
        "aggressive_testing",
        "mixed_conditions"
    ] = "baseline",
    duration: float = 600,
) -> None:
    env = simpy.Environment()
    office = create_office_simulation(env, scenario)
    env.run(until=duration)

    print_dashboard(office, scenario, duration)


In [16]:
run_simulation("baseline", duration=500)


  BASELINE

  Statistics
  H  9  ####
  I  0  
  S 18  #########
  R 23  ###########

  Infected: 41/50 (82%)
  Peak:     41 at t=255

  Timeline Heatmap
  H |████████▓▒▒::···································|
  I |        ·::░░▒▒░░::···                          |
  S |             ··::░░░▒▒▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▒▒▒░░░░░░:|
  R |                                      ····::::::|
    ------------------------------------------------
    0         125         250         375         500

  Key Events
        ● First sick (t=75)
            ◆ 50% infected (t=110)
                         ▲ Peak sick (41) (t=255)
                                ○ First recovery (t=335)
  --------------------------------------------------

  Intensity: ' ' = None   '·' = Low   '█' = High

